# Analysis for the unconditional generation

In [ ]:
import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns
from matplotlib import pyplot as plt
from rich.progress import track
from plotly import express as px
from tqdm import tqdm

In [ ]:
from tools import count_novel_elements, count_unique_elements, count_unique_novel_elements

## Load data

In [ ]:
dtypes = {
    "loads": "boolean",
    "errorfree": "boolean",
    "sanitizes": "boolean",
    "connected": "boolean",
    "all_hydrogens": "boolean",
    "no_radicals": "boolean",
    "chemical": "boolean",
    "physical": "boolean",
    "fail": "boolean",
}

In [ ]:
dfs = {}
results = {
    "EQGAT": "predictions/unconditional/eqgat/eqgat_100000_predictions.csv",
    "GCDM": "predictions/unconditional/gcdm/gcdm_100000_predictions.csv",
    "GeoLDM": "predictions/unconditional/geoldm/geoldm_100000_predictions.csv",
    "SemlaFlow": "predictions/unconditional/semlaflow/semlaflow_100000_predictions.csv",
    "FlowMol": "predictions/unconditional/flowmol/flowmol_100000_predictions.csv",
}
# results_opt = {
#     "EQGAT + PP": "predictions/unconditional/eqgat/eqgat_100000_predictions_optimized.csv",
#     "GCDM + PP": "predictions/unconditional/gcdm/gcdm_100000_predictions_optimized.csv",
#     "GeoLDM + PP": "predictions/unconditional/geoldm/geoldm_100000_predictions_optimized.csv",
#     "SemlaFlow + PP": "predictions/unconditional/semlaflow/semlaflow_100000_predictions_optimized.csv",
#     "FlowMol + PP": "predictions/unconditional/flowmol/flowmol_100000_predictions_optimized.csv",
# }
datasets = {
    "GEOM Drugs": "data/unconditional/geom-drugs/all.csv",
    # "GEOM Drugs Training": "data/unconditional/geom-drugs/train.csv",
    # "GEOM Drugs Validation": "predictions_data/val.csv",
    # "GEOM Drugs Testing": "predictions_data/test.csv",
    # "DrugBank All": "data/unconditional/drugbank/all_structures_3d.csv",
    "DrugBank Approved": "data/unconditional/drugbank/approved_structures_2d.csv",
}
for name, file in track((results | results_opt | datasets).items()):
    df = pd.read_csv(file, low_memory=False, dtype=dtypes)
    df["method"] = name
    if "fail" not in df:
        df["fail"] = False
    else:
        df["fail"] = df["fail"].fillna(False)
    dfs[name] = df
df = pd.concat(dfs.values()).reset_index(drop=True)

In [ ]:
# define order of methods
order = [
    # "GEOM Drugs Validation",
    # "GEOM Drugs Testing",
    "FlowMol",
    "SemlaFlow",
    "EQGAT",
    "GCDM",
    "GeoLDM",
    "FlowMol + PP",
    "SemlaFlow + PP",
    "EQGAT + PP",
    "GCDM + PP",
    "GeoLDM + PP",
    "GEOM Drugs",
    # "GEOM Drugs training data",
    "DrugBank All",
    # "DrugBank Approved",
]
df["method"] = pd.Categorical(df["method"], categories=order, ordered=True)

In [ ]:
df.columns

In [ ]:
# enrichment
reference = "GEOM Drugs"
df_ref = dfs[reference]
reference_smiles = set(df_ref["smiles"].values)
df["total_number"] = True
df["chemical"] = df["loads"] & df["sanitizes"] & df["connected"] & df["no_radicals"] & df["all_hydrogens"]
df["valid"] = df["connected"] & df["chemical"] & df["physical"]
df["novel"] = df["smiles"].map(lambda x: x not in reference_smiles)
df["valid_novel"] = df["valid"] & df["novel"]

In [ ]:
df.loc[df["smiles"].isna(), "smiles"] = pd.NA
df["valid_smiles"] = pd.NA
df.loc[df.valid, "valid_smiles"] = df.loc[df.valid, "smiles"]
df.loc[df.valid_smiles.isna(), "valid"] = False
assert df["valid_smiles"].notna().sum() == df["valid"].sum()

In [ ]:
df["delta"] = df["mol_pred_energy"] - df["ensemble_avg_energy"]
df["delta_normalized"] = df["delta"] / df["ensemble_avg_energy"]


In [ ]:
metrics = {
    "ensemble_avg_energy": "Ensemble Average Energy",
    "mol_pred_energy": "Molecular Prediction Energy",
    "energy_ratio": "Energy Ratio",
    "delta": "Energy delta",
    "delta_normalized": "Energy delta (normalized)",
    "sa": "Synthetic Accessability Score",
    "sa_normalized": "Synthetic Accessability Score (normalized)",
    "spacial": "Spacial Score",
    "qed": "Quantitative Estimation of Drug-likeness",
    "logp": "LogP",
    "lipinski": "Lipinski Rule of 5",
    "num_heavy": "Number of Heavy Atoms",
    "weight": "Molecular Weight",
    "num_rings": "Number of Rings",
}

In [ ]:
df

## Check

In [ ]:
df.groupby("method").size()

In [ ]:
df.head()

## Tables

In [ ]:
def wrap_envs(latex: str, envs: list[str] = ["center", "small", "sc"]) -> str:
    for env in envs:
        latex = latex.replace(r"\begin{tabular}", r"\begin{" + env + "}" + "\n" + r"\begin{tabular}")
        latex = latex.replace(r"\end{tabular}", r"\end{tabular}" + "\n" + r"\end{" + env + "}")
    return latex


In [ ]:
n = 100000


def denominator(x):
    if len(x) == 0:
        return float("nan")
    # generated molecules
    if (len(x) > 20000) and (len(x) <= n):
        return n
    # training data and drugbank
    return len(x)


def mean(x):
    return np.sum(x) / denominator(x)


def std(x):
    return np.sqrt(np.sum((x - mean(x)) ** 2) / denominator(x))


### Validity, Uniqueness, and Novelty

In [ ]:
n = 100000


df_filter = df
aggs = {
    "Number": ("total_number", "sum"),
    r"\unit{\percent}  Valid": (
        "valid",
        lambda x: sum(x) / denominator(x),
    ),
    # r"\unit{\percent}  Unique": (
    #     "smiles",
    #     lambda x: count_unique_elements(x) / denominator(x),
    # ),
    #  r"\unit{\percent}  Novel": (
    #     "smiles",
    #     lambda x: count_novel_elements(x, reference_smiles )  / denominator(x),
    # ),
    # r"\unit{\percent}  Novel \&  Unique": (
    #     "smiles",
    #     lambda x: count_unique_novel_elements(x, reference_smiles ) / denominator(x),
    # ),
    # r"\unit{\percent}  Novel \&  Unique \& Valid": (
    #     "valid_smiles",
    #     lambda x: count_unique_novel_elements(x, reference_smiles )  / denominator(x),
    # ),
    r"\unit{\percent}  Valid \&  Unique": (
        "valid_smiles",
        lambda x: count_unique_elements(x) / denominator(x),
    ),
    # r"\unit{\percent}  Valid \& Novel": (
    #     "valid_smiles",
    #     lambda x: compute_novelty(x, reference_smiles, total=1)  / denominator(x),
    # ),
    r"\unit{\percent}  Valid \& Unique \& Novel": (
        "valid_smiles",
        lambda x: count_unique_novel_elements(x, reference_smiles) / denominator(x),
    ),
}
df_agg = df_filter.groupby("method", observed=False).agg(**aggs)
cols = df_agg.columns
df_style = (
    df_agg.style.format("{:.0f}", subset=cols[:1]).format("{:.1%}", subset=cols[1:], escape="latex").format_index(escape="latex", axis=0)
)
df_style.index.name = None
df_style

In [ ]:
caption = """
Validity, uniqueness, and novelty of 100k unconditionally generated molecules.
The table contains the percentages of molecules that are valid, valid and unique, and valid, unique, and novel, out of the total number generated or contained in the data set.
"""
label = "tab:unconditional_novelty"
cols_def = "lS[table-format=3.1]S[table-format=3.1]S[table-format=3.1]"
latex = df_style.to_latex(
    label=label,
    siunitx=True,
    environment="table*",
    hrules=True,
    caption=caption,
    column_format=cols_def,
)
latex = latex.replace("%", "").replace("nan", "")
latex = wrap_envs(latex, ["center"])
latex = latex.replace("GEOM", r"\midrule" "\nGEOM")
latex = latex.replace("FlowMol Optimized", r"\midrule" "\nFlowMol Optimized")
with open("tables/unconditional_novelty.tex", "w") as f:
    f.write(latex)

### Validity

In [ ]:
aggs = {
    "Number": ("total_number", "sum"),
    r"\unit{\percent} Generated": ("total_number", mean),
    # r"\unit{\percent} Connected": ("connected", mean),
    r"\unit{\percent} Chemical": ("chemical", mean),
    r"\unit{\percent} Physical": ("physical", mean),
    r"\unit{\percent} Valid": ("valid", mean),
}
df_agg = df.groupby("method", observed=False).agg(**aggs)
cols = df_agg.columns
# df_agg.iloc[-3, 1] = pd.NA
df_agg.iloc[-2, 1] = pd.NA
df_agg.iloc[-1, 1] = pd.NA
df_style = (
    df_agg.style.format("{:.0f}", subset=cols[:1]).format("{:.1%}", subset=cols[1:], escape="latex").format_index(escape="latex", axis=0)
)
df_style.index.name = None
df_style

In [ ]:
caption = """
Validity of the generated and ground truth molecules.
The table contains the total number of molecules that was to be generated or is contained in the data set and the proportions of molecules that pass all the chemical validity test, all physical validity tests.
The last columns is the percentage of molecules that pass all the tests together.
"""
label = "tab:unconditional_validity"
cols_def = "lS[table-format=6.0]S[table-format=3.1]S[table-format=3.1]S[table-format=3.1]|S[table-format=3.1]"
latex = df_style.to_latex(label=label, siunitx=True, environment="table*", hrules=True, caption=caption, column_format=cols_def)
latex = latex.replace("%", "").replace("nan", "")
latex = wrap_envs(latex, ["center"])
latex = latex.replace("GEOM", r"\midrule" "\nGEOM")
latex = latex.replace("FlowMol Optimized", r"\midrule" "\nFlowMol Optimized")
with open("tables/unconditional_validity.tex", "w") as f:
    f.write(latex)

### Physical details

In [ ]:
cols_fillna = ["bond_lengths", "bond_angles", "internal_steric_clash", "aromatic_ring_flatness", "double_bond_flatness", "internal_energy"]
aggs = {
    # "Total number": ("total_number", "sum"),
    # r"\unit{\percent} InChI": ("inchi_convertible", mean),
    r"Bond lengths": ("bond_lengths", mean),
    r"Bond angles": ("bond_angles", mean),
    r"Internal steric clash": ("internal_steric_clash", mean),
    r"Aromatic ring flatness": ("aromatic_ring_flatness", mean),
    r"Double bond flatness": ("double_bond_flatness", mean),
    r"Internal energy": ("internal_energy", mean),
    r"\unit{\percent} Physical": ("physical", mean),
}
df[cols_fillna] = df[cols_fillna].fillna(False)
df["physical"] = df[cols_fillna].all(axis=1)
df_agg = df.groupby("method", observed=False).agg(**aggs)
cols = df_agg.columns
df_style = df_agg.style.format("{:.1%}", subset=cols[0:], escape="latex").format_index(escape="latex", axis=0)
df_style.index.name = None
df_style

In [ ]:
caption = """
Components of the physical validity of the generated and ground truth molecules.
The numbers shown are the percentages of the molecules that pass each of the intramolecular PoseBusters tests.
The last column is the percentage of molecules that pass all of the intramolecular tests.
The physical tests check the 3D conformations of the generated molecules.
"""
label = "tab:physical_validity"
cols_def = "l" + "S[table-format=3.1]" * 6 + "|S[table-format=3.1]"
latex = df_style.to_latex(label=label, siunitx=True, environment="table*", hrules=True, caption=caption, column_format=cols_def)
latex = latex.replace("%", "").replace("nan", "")
# latex = wrap_envs(latex, ["center"])
latex = latex.replace("GEOM", r"\midrule\n GEOM")
latex = latex.replace("FlowMol Optimized", r"\midrule\n FlowMol Optimized")
col1 = r"{} & {Bond lengths} & {Bond angles} & {Internal steric clash} & {Aromatic ring flatness} & {Double bond flatness} & {Internal energy} & {\unit{\percent} Physical} \\"
col2 = (
    r"{} & {Bond} & {Bond} & {Internal} & {Aromatic ring} & {Double bond} & {Internal} & {} \\"
    "\n"
    r"{} & {lengths} & {angles} & {steric clash} & {flatness} & {flatness} & {energy} & {\unit{\percent} Physical} \\"
)
latex = latex.replace(col1, col2)
with open("tables/unconditional_physical_validity.tex", "w") as f:
    f.write(latex)

### Chemical details

In [ ]:
cols_fillna = ["all_atoms_connected", "mol_pred_loaded", "sanitization", "no_h_added", "inchi_convertible"]
aggs = {
    "Number": ("total_number", "sum"),
    # r"\unit{\percent} InChI": ("inchi_convertible", mean),
    # r"File read": ("mol_pred_loaded", mean),
    r"Hydrogen complete": ("no_h_added", mean),
    r"Sanitizes": ("sanitization", mean),
    r"InChI convertible": ("inchi_convertible", mean),
    r"Connectedness": ("all_atoms_connected", mean),
    r"\unit{\percent} Chemical": ("chemical", mean),
}
df[cols_fillna] = df[cols_fillna].fillna(False)
df["chemical"] = df[cols_fillna].all(axis=1)
df_agg = df.groupby("method", observed=False).agg(**aggs)
cols = df_agg.columns
df_style = (
    df_agg.style.format("{:.0f}", subset=cols[:1]).format("{:.1%}", subset=cols[1:], escape="latex").format_index(escape="latex", axis=0)
)
df_style.index.name = None
df_style

In [ ]:
caption = """
Components of the chemical validity of the generated and ground truth molecules. The numbers shown are the
percentages of the molecules that pass each of the tests. The last column is the percentage of
molecules that pass all of the tests.
The chemical tests check the molecular graph generated and they do not check the molecules' 3D conformations.
"""
label = "tab:chemical_validity"
cols_def = "l" + "S[table-format=7]" + "S[table-format=3.1]" * 4 + "|S[table-format=3.1]"
latex = df_style.to_latex(label=label, siunitx=True, environment="table*", hrules=True, caption=caption, column_format=cols_def)
latex = latex.replace("%", "").replace("nan", "")
# latex = wrap_envs(latex, ["center"])
latex = latex.replace("GEOM", r"\midrule" "\nGEOM")
latex = latex.replace("FlowMol + PP", r"\midrule" "\nFlowMol + PP")
with open("tables/unconditional_chemical_validity.tex", "w") as f:
    f.write(latex)

### Properties of valid molecules

In [ ]:
# combine mean and std into one column
names = [
    "weight",
    "num_heavy",
    "num_rings",
    "logp",
    # "sa",
    # "qed",
    # "energy_ratio",
    # "lipinski",
    # "spacial",
]
columns = {}
for column in set(c[0] for c in df_agg.columns if c[0] in names):
    columns[metrics[column]] = df_agg[column].apply(lambda x: f"{x[0]:.2f}({x[1]:.2f})", axis=1)
df_style = pd.DataFrame(columns)
# df_style = df_style[[names[c] for c in df_style.columns]]
df_style = df_style.style.format_index(escape="latex", axis=1)
df_style.index.name = None

caption = """
Various molecular properties.
The table contains the QED, SA, energy ratio, weight, number of heavy atoms, number of rings, Lipinski rule of five, logP, and spacial metrics.
"""
label = "tab:unconditional_properties"
latex = df_style.to_latex(
    label=label,
    siunitx=True,
    environment="table*",
    hrules=True,
    caption=caption,
    column_format="l" + "S[table-format=3.2(5)]S[table-format=3.2(3)]S[table-format=3.2(3)]S[table-format=3.2(3)]",
)
latex = latex.replace("%", "").replace("nan(nan)", "").replace("nan", "")
latex = wrap_envs(latex, ["center"])
latex = latex.replace(r"\begin{center}", r"\begin{center}" + "\n" + "\sisetup{separate-uncertainty}")
latex = latex.replace("GEOM", r"\midrule\n GEOM")
latex = latex.replace("FlowMol Optimized", r"\midrule\n FlowMol Optimized")
with open("tables/unconditional_properties_2.tex", "w") as f:
    f.write(latex)

In [ ]:
# combine mean and std into one column
names = [
    # "weight",
    # "num_heavy",
    # "num_rings",
    # "logp",
    "qed",
    "sa",
    "energy_ratio",
    "lipinski",
    "spacial",
]
columns = {}
for column in set(c[0] for c in df_agg.columns if c[0] in names):
    columns[metrics[column]] = df_agg[column].apply(lambda x: f"{x[0]:.2f}({x[1]:.2f})", axis=1)
df_style = pd.DataFrame(columns)
# df_style = df_style[[names[c] for c in df_style.columns]]
df_style = df_style.style.format_index(escape="latex", axis=1)
df_style.index.name = None

caption = """
Various molecular properties.
The table contains the QED, SA, energy ratio, weight, number of heavy atoms, number of rings, Lipinski rule of five, logP, and spacial metrics.
"""
label = "tab:unconditional_properties"
latex = df_style.to_latex(
    label=label,
    siunitx=True,
    environment="table*",
    hrules=True,
    caption=caption,
    column_format="l" + "S[table-format=3.2(3)]S[table-format=3.2(3)]S[table-format=3.2(4)]S[table-format=3.2(3)]S[table-format=3.2(4)]",
)
latex = latex.replace("%", "").replace("nan(nan)", "").replace("nan", "")
latex = wrap_envs(latex, ["center"])
latex = latex.replace(r"\begin{center}", r"\begin{center}" + "\n" + r"\sisetup{separate-uncertainty}")
latex = latex.replace("GEOM", r"\midrule" "\n GEOM")
latex = latex.replace("FlowMol Optimized", r"\midrule" "\n FlowMol Optimized")
with open("tables/unconditional_properties_1.tex", "w") as f:
    f.write(latex)

### Energy ratio

In [ ]:
def interquartile_range(s: pd.Series) -> float:
    s = sorted(s.dropna())
    n = len(s)
    if n == 0:
        return float("nan")
    q1 = s[n // 4]
    q3 = s[(3 * n) // 4]
    return q3 - q1


# df_filter = df[df.valid]
df_filter = df
aggs = {
    # "ensemble_avg_energy": ["mean", "std"],
    # "mol_pred_energy": ["mean", "std"],
    # "energy_ratio": ["mean", "std"],
    "ensemble_avg_energy": ["median", interquartile_range],
    "mol_pred_energy": ["median", interquartile_range],
    "energy_ratio": ["median", interquartile_range],
}
df_agg = df_filter.groupby("method", observed=False).agg(aggs)
cols = df_agg.columns
# df_style = df_agg.style.format("{:.2f}", subset=cols).format_index(
#     escape="latex", axis=1
# )
df_style = df_agg
df_style.index.name = None
df_style

In [ ]:
# caption = """
# Energy ratio.
# The table contains the ensemble average energy, the predicted energy, and the energy ratio.
# """
# label = "tab:unconditional_energy"
# latex = df_style.to_latex(
#     label=label, siunitx=True, environment="table*", hrules=True, caption=caption
# )
# latex = latex.replace("%", "").replace("nan", "")
# with open("tables/unconditional_energy.tex", "w") as f:
#     f.write(latex)

## Plots

In [ ]:
# define order of methods
order_data = [
    "DrugBank All",
    "GEOM Drugs",
]
order_methods = [
    "FlowMol",
    "SemlaFlow",
    "EQGAT",
    "GCDM-SBDD",
    "GeoLDM",
    # "FlowMol Optimized",
    # "SemlaFlow Optimized",
    # "EQGAT Optimized",
    # "GCDM-SBDD Optimized",
    # "GeoLDM Optimized",
]

In [ ]:
def make_histplot(df, metric, **kwargs):
    # sns.histplot(
    #     df[(df.valid) & (df.method.isin(order_methods + order_data))][["method", metric]].reset_index(drop=True),
    #     x=metric,
    #     hue="method",
    #     cumulative=False,
    #     common_norm=False,
    #     stat="density",
    #     element="step",
    #     hue_order=order_data + order_methods,
    #     linewidth=0.0,
    #     fill=False,
    #     **kwargs,
    #     # legend=True, palette="tab10", linewidth=1.5
    # )
    sns.kdeplot(
        df[(df.valid) & (df.method.isin(order_data))][["method", metric]].reset_index(drop=True),
        x=metric,
        hue="method",
        cumulative=False,
        common_norm=False,
        # stat="density",
        # element="step",
        # kde=True,
        hue_order=order_data,
        linewidth=1.5,
        fill=True,
        **kwargs,
        # legend=True, palette="tab10", linewidth=1.5
    )
    sns.kdeplot(
        df[(df.valid) & (df.method.isin(order_methods))][["method", metric]].reset_index(drop=True),
        x=metric,
        hue="method",
        cumulative=False,
        common_norm=False,
        # stat="density",
        # element="step",
        # kde=True,
        hue_order=order_data + order_methods,
        linewidth=1.5,
        fill=False,
        **kwargs,
        # legend=True, palette="tab10", linewidth=1.5
    )

### Synthetically accessibility

In [ ]:
metric = "sa"
name = metrics[metric]
make_histplot(df, metric)  # , bins=40)
# plt.title(name)
plt.xlabel(name)
plt.xlim(0, 10)
plt.savefig(f"plots/unconditional_{metric}.png")

### QED

In [ ]:
metric = "qed"
name = metrics[metric]
make_histplot(df, metric)  # , bins=50)
# plt.title(name)
plt.xlabel(name)
plt.xlim(0, 1)
plt.savefig(f"plots/unconditional_{metric}.png")

### Energy ratio

In [ ]:
metric = "energy_ratio"
name = metrics[metric]
make_histplot(df, metric, log_scale=True)  # , bins=50)
# plt.title(name)
plt.xlabel(name)
plt.xlim(0.5, 150)
plt.savefig(f"plots/unconditional_{metric}.png")

In [ ]:
metric = "delta"
name = metrics[metric]
make_histplot(df, metric, log_scale=True)  # , bins=20)
# plt.title(name)
plt.xlabel(name)
plt.xlim(0.1, 1e5)
plt.savefig(f"plots/unconditional_{metric}.png")

In [ ]:
metric = "delta_normalized"
name = metrics[metric]
make_histplot(df, metric, log_scale=True)  # , bins=20)
# plt.title(name)
plt.xlabel(name)
plt.xlim(0.01, 1000)
plt.savefig(f"plots/unconditional_{metric}.png")

### Number of heavy atoms

In [ ]:
metric = "num_heavy"
name = metrics[metric]
make_histplot(df, metric)  # , bins=20)
# plt.title(name)
plt.xlabel(name)
plt.xlim(0, 60)
plt.savefig(f"plots/unconditional_{metric}.png")

In [ ]:
metric = "spacial"
name = metrics[metric]
make_histplot(df, metric)  # , bins=20)
# plt.title(name)
plt.xlabel(name)
plt.xlim(0, 70)
plt.savefig(f"plots/unconditional_{metric}.png")

# SI

## Plots

In [ ]:
metric = "logp"
name = metrics[metric]
make_histplot(df, metric)  # , bins=20)
# plt.title(name)
plt.xlabel(name)
# plt.xlim(0, 60)
plt.savefig(f"plots/unconditional_{metric}.png")

In [ ]:
metric = "weight"
name = metrics[metric]
make_histplot(df, metric)  # , bins=20)
# plt.title(name)
plt.xlabel(name)
plt.xlim(0, 1000)
plt.savefig(f"plots/unconditional_{metric}.png")

In [ ]:
metric = "lipinski"
name = metrics[metric]
make_histplot(df, metric)  # , bins=20)
# plt.title(name)
plt.xlabel(name)
plt.xlim(0.1, 5.9)
plt.savefig(f"plots/unconditional_{metric}.png")

In [ ]:
metric = "num_rings"
name = metrics[metric]
make_histplot(df, metric)  # , bins=20)
# plt.title(name)
plt.xlabel(name)
plt.xlim(0.1, 14.9)
plt.savefig(f"plots/unconditional_{metric}.png")

### Physcial check details

In [ ]:
# define order of methods
order_data = [
    "DrugBank All",
    "GEOM Drugs",
]
order_methods = [
    "FlowMol",
    "SemlaFlow",
    "EQGAT",
    "GCDM-SBDD",
    "GeoLDM",
    # "FlowMol Optimized",
    # "SemlaFlow Optimized",
    # "EQGAT Optimized",
    # "GCDM-SBDD Optimized",
    # "GeoLDM Optimized",
]

In [ ]:
metric_names = {
    "shortest_noncovalent_relative_distance": "Shortest normalized non-covalent distance",
    "shortest_bond_relative_length": "Shortest bond normalized bond length",
    "longest_bond_relative_length": "Longest bond normalized bond length",
    "most_extreme_relative_angle": "Most extreme normalized bond angle",
}

In [ ]:
def plot_violations(df, metric, figsize=(5, 10)):
    dfx = df.copy()
    dfx = dfx[np.isfinite(dfx[metric])]
    dfx = dfx[dfx.method.isin(order_data + order_methods)]
    methods_present = [o for o in order_data + order_methods if o in set(dfx.method)]
    dfx["method"] = pd.Categorical(dfx["method"], categories=methods_present, ordered=True)
    df_plot = dfx.reset_index()

    f, ax = plt.subplots(figsize=figsize)

    sns.boxplot(
        df_plot,
        x=metric,
        y="method",
        hue="method",
        hue_order=order_data + order_methods,
        showfliers=False,
        ax=ax,
        # boxprops=dict(alpha=0.5),
    )
    # for patch in ax.artists:
    #     fc = patch.get_facecolor()
    #     patch.set_facecolor(plt.colors.to_rgba(fc, 0.1))

    for collection in plt.gca().collections:
        collection.set_facecolor([*collection.get_facecolor()[0][:3], 0.3])  # Set alpha to 0.3
        collection.set_edgecolor([*collection.get_edgecolor()[0][:3], 0.3])  # Set alpha to 1.0

    sns.stripplot(
        df_plot,
        x=metric,
        y="method",
        hue="method",
        hue_order=order_data + order_methods,
        dodge=False,
        ax=ax,
        size=5,
        linewidth=1,
        alpha=1.0,
    )

    print(len(plt.gca().collections))

    # for collection in plt.gca().collections:
    #     collection.set_facecolor([*collection.get_facecolor()[0][:3], 0.3])  # Set alpha to 0.3
    #     collection.set_edgecolor([*collection.get_edgecolor()[0][:3], 1.0])  # Set alpha to 1.0

    # move strip plot down
    # for i, artist in enumerate(ax.collections):
    #     artist.set_offsets(artist.get_offsets() + [0, 0.5])

    plt.xlabel(metric_names[metric])
    ax.set_ylabel("")


In [ ]:
metric = "shortest_noncovalent_relative_distance"
plot_violations(df, metric, figsize=(7, 5))
plt.axvline(x=0.75, color="red", linestyle="--", linewidth=1, zorder=0)
plt.axvspan(0.4, 0.75, color="red", alpha=0.05, zorder=0)
plt.xlim(0.4, 1.2)
plt.savefig(f"plots/unconditional_physcheck_{metric}.png", bbox_inches="tight")

In [ ]:
metric = "shortest_bond_relative_length"
plot_violations(df, metric, figsize=(7, 5))
plt.axvline(x=0.8, color="red", linestyle="--", linewidth=1, zorder=0)
plt.axvspan(0.5, 0.8, color="red", alpha=0.05, zorder=0)
plt.xlim(0.5, 1.2)
plt.savefig(f"plots/unconditional_physcheck_{metric}.png", bbox_inches="tight")

In [ ]:
metric = "longest_bond_relative_length"
plot_violations(df, metric, figsize=(7, 5))
plt.axvline(x=1.2, color="red", linestyle="--", linewidth=1, zorder=0)
plt.axvspan(1.2, 1.4, color="red", alpha=0.05, zorder=0)
plt.xlim(0.9, 1.4)
plt.savefig(f"plots/unconditional_physcheck_{metric}.png", bbox_inches="tight")

In [ ]:
metric = "most_extreme_relative_angle"
plot_violations(df, metric, figsize=(7, 5))
plt.axvline(x=1.2, color="red", linestyle="--", linewidth=1, zorder=0)
plt.axvspan(1.2, 1.5, color="red", alpha=0.05, zorder=0)
plt.xlim(0.95, 1.5)
plt.savefig(f"plots/unconditional_physcheck_{metric}.png", bbox_inches="tight")

#### Waterfall

In [ ]:
cols_dict = {
    "fail": "Generated",
    "chemical": "Chemical",
    "physical": "Physical",
    "valid": "Valid",
}

In [ ]:
df.groupby("method")[list(cols_dict)].mean()